In [ ]:
import ee

ee.Authenticate(force=True)


In [ ]:
import ee

ee.Initialize(project="gis-project-work")

print("✅ Earth Engine initialized successfully!")

In [ ]:
import ee
import pandas as pd
import os

# -----------------------------------------
# 1. Initialize Earth Engine
# -----------------------------------------

ee.Initialize(project="gis-project-work")

print("✅ Earth Engine initialized")


# -----------------------------------------
# 2. Joshimath
# -----------------------------------------

joshimath = ee.Geometry.Point([
    79.56436,   # longitude
    30.55543    # latitude
])

# 10 km buffer
roi = joshimath.buffer(10000)

print("✅ Joshimath 10 km area created")


# -----------------------------------------
# 3. GPM IMERG V07
# -----------------------------------------

gpm = ee.ImageCollection(
    "NASA/GPM_L3/IMERG_V07"
).filterBounds(roi).select("precipitation")

print("✅ GPM dataset loaded")


# -----------------------------------------
# 4. Create folder
# -----------------------------------------

output_folder = "data/rainfall_monthly"

os.makedirs(output_folder, exist_ok=True)


# -----------------------------------------
# 5. Download each month separately
# -----------------------------------------

for year in range(2020, 2027):

    for month in range(1, 13):

        # Stop after August 2026
        if year == 2026 and month > 8:
            break

        print(f"\n📥 Processing {year}-{month:02d}...")

        # Start and end of month
        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, "month")

        # Filter GPM
        monthly = (
            gpm
            .filterDate(start, end)
        )

        # Get number of observations
        count = monthly.size().getInfo()

        print(f"   GPM observations: {count}")

        # -----------------------------------------
        # Extract every observation
        # -----------------------------------------

        def extract_image(image):

            # Calculate mean rainfall over 10 km Joshimath area
            value = image.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=roi,
                scale=11132,
                maxPixels=1e9
            ).get("precipitation")

            return ee.Feature(
                None,
                {
                    "datetime": image.date().format(
                        "YYYY-MM-dd HH:mm:ss"
                    ),
                    "rainfall_mm_per_hour": value
                }
            )

        features = monthly.map(extract_image)

        # -----------------------------------------
        # Get data from GEE
        # -----------------------------------------

        data = features.getInfo()

        rows = []

        for feature in data["features"]:

            props = feature["properties"]

            rows.append({
                "datetime": props["datetime"],
                "rainfall_mm_per_hour": props[
                    "rainfall_mm_per_hour"
                ]
            })

        # -----------------------------------------
        # Pandas DataFrame
        # -----------------------------------------

        df = pd.DataFrame(rows)

        # Add year/month columns
        df["year"] = year
        df["month"] = month

        # Sort by datetime
        df = df.sort_values("datetime").reset_index(drop=True)

        # -----------------------------------------
        # Save separate CSV
        # -----------------------------------------

        filename = (
            f"{output_folder}/"
            f"rainfall_{year}_{month:02d}.csv"
        )

        df.to_csv(
            filename,
            index=False
        )

        print(f"   ✅ Saved: {filename}")
        print(f"   Rows: {len(df)}")


print("\n🎉 ALL MONTHLY RAINFALL FILES DOWNLOADED!")

In [ ]:
import ee
import pandas as pd
import os

# =========================================================
# 1. Initialize Earth Engine
# =========================================================

ee.Initialize(project="gis-project-work")

print("✅ Earth Engine initialized")


# =========================================================
# 2. Joshimath 10 km study area
# =========================================================

joshimath = ee.Geometry.Point([
    79.56436,   # longitude
    30.55543    # latitude
])

roi = joshimath.buffer(10000)

print("✅ Joshimath 10 km area created")


# =========================================================
# 3. SMAP Soil Moisture dataset
# =========================================================

smap = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
    .filterBounds(roi)
    .select("soil_moisture_am")
)

print("✅ SMAP dataset loaded")


# =========================================================
# 4. Output folder
# =========================================================

output_folder = "data/soil_moisture_monthly"

os.makedirs(output_folder, exist_ok=True)


# =========================================================
# 5. Function to extract soil moisture
# =========================================================

def extract_soil_moisture(image):

    value = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=9000,
        maxPixels=1e9
    ).get("soil_moisture_am")

    return ee.Feature(
        None,
        {
            "datetime": image.date().format(
                "YYYY-MM-dd HH:mm:ss"
            ),
            "soil_moisture": value
        }
    )


# =========================================================
# 6. Download month-by-month
# =========================================================

for year in range(2020, 2027):

    for month in range(1, 13):

        # Stop after August 2026
        if year == 2026 and month > 8:
            break

        print(f"\n📥 Processing {year}-{month:02d}...")

        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, "month")

        # Filter month
        monthly = smap.filterDate(start, end)

        image_count = monthly.size().getInfo()

        print(f"   SMAP images: {image_count}")

        # -------------------------------------------------
        # Extract values
        # -------------------------------------------------

        features = monthly.map(extract_soil_moisture)

        # Remove images where soil moisture is missing
        features = features.filter(
            ee.Filter.notNull(["soil_moisture"])
        )

        # Get data
        data = features.getInfo()

        rows = []

        for feature in data["features"]:

            props = feature["properties"]

            rows.append({
                "datetime": props["datetime"],
                "soil_moisture": props["soil_moisture"],
                "year": year,
                "month": month
            })

        # -------------------------------------------------
        # Create DataFrame
        # -------------------------------------------------

        df = pd.DataFrame(rows)

        if not df.empty:

            df = df.sort_values(
                "datetime"
            ).reset_index(drop=True)

        # -------------------------------------------------
        # Save separate CSV
        # -------------------------------------------------

        filename = (
            f"{output_folder}/"
            f"soil_moisture_{year}_{month:02d}.csv"
        )

        df.to_csv(
            filename,
            index=False
        )

        print(f"   ✅ Saved: {filename}")
        print(f"   Valid observations: {len(df)}")


print("\n🎉 ALL MONTHLY SOIL-MOISTURE FILES COMPLETED!")

In [ ]:
import ee
import pandas as pd
import os

# =========================================================
# 1. Initialize Earth Engine
# =========================================================

ee.Initialize(project="gis-project-work")

print("✅ Earth Engine initialized")


# =========================================================
# 2. Joshimath 10 km study area
# =========================================================

joshimath = ee.Geometry.Point([
    79.56436,   # longitude
    30.55543    # latitude
])

roi = joshimath.buffer(10000)

print("✅ Joshimath 10 km study area created")


# =========================================================
# 3. DEM
# =========================================================
# SRTM elevation dataset
# Elevation is in meters
# =========================================================

dem = ee.Image("USGS/SRTMGL1_003").select("elevation")

print("✅ DEM loaded")


# =========================================================
# 4. Calculate Slope
# =========================================================

slope = ee.Terrain.slope(dem)

print("✅ Slope calculated")


# =========================================================
# 5. Output folders
# =========================================================

dem_folder = "data/DEM_monthly"
slope_folder = "data/Slope_monthly"

os.makedirs(dem_folder, exist_ok=True)
os.makedirs(slope_folder, exist_ok=True)


# =========================================================
# 6. Extract DEM and slope for every month
# =========================================================

for year in range(2020, 2027):

    for month in range(1, 13):

        # Stop after August 2026
        if year == 2026 and month > 8:
            break

        print(f"\n📥 Processing {year}-{month:02d}...")


        # =================================================
        # DEM statistics
        # =================================================

        dem_stats = dem.reduceRegion(
            reducer=ee.Reducer.mean()
                .combine(
                    reducer2=ee.Reducer.min(),
                    sharedInputs=True
                )
                .combine(
                    reducer2=ee.Reducer.max(),
                    sharedInputs=True
                ),
            geometry=roi,
            scale=30,
            maxPixels=1e9
        ).getInfo()


        # =================================================
        # Slope statistics
        # =================================================

        slope_stats = slope.reduceRegion(
            reducer=ee.Reducer.mean()
                .combine(
                    reducer2=ee.Reducer.min(),
                    sharedInputs=True
                )
                .combine(
                    reducer2=ee.Reducer.max(),
                    sharedInputs=True
                ),
            geometry=roi,
            scale=30,
            maxPixels=1e9
        ).getInfo()


        # =================================================
        # DEM DataFrame
        # =================================================

        dem_df = pd.DataFrame([{
            "year": year,
            "month": month,
            "elevation_mean_m": dem_stats.get(
                "elevation_mean"
            ),
            "elevation_min_m": dem_stats.get(
                "elevation_min"
            ),
            "elevation_max_m": dem_stats.get(
                "elevation_max"
            )
        }])


        # =================================================
        # Slope DataFrame
        # =================================================

        slope_df = pd.DataFrame([{
            "year": year,
            "month": month,
            "slope_mean_degree": slope_stats.get(
                "slope_mean"
            ),
            "slope_min_degree": slope_stats.get(
                "slope_min"
            ),
            "slope_max_degree": slope_stats.get(
                "slope_max"
            )
        }])


        # =================================================
        # Save DEM
        # =================================================

        dem_file = (
            f"{dem_folder}/"
            f"DEM_{year}_{month:02d}.csv"
        )

        dem_df.to_csv(
            dem_file,
            index=False
        )


        # =================================================
        # Save Slope
        # =================================================

        slope_file = (
            f"{slope_folder}/"
            f"Slope_{year}_{month:02d}.csv"
        )

        slope_df.to_csv(
            slope_file,
            index=False
        )


        print(f"   ✅ DEM saved: {dem_file}")
        print(f"   ✅ Slope saved: {slope_file}")


print("\n🎉 DEM AND SLOPE DATA COMPLETED!")

In [ ]:
import ee
import pandas as pd
import os

# =========================================================
# 1. Initialize Earth Engine
# =========================================================

ee.Initialize(project="gis-project-work")

print("✅ Earth Engine initialized")


# =========================================================
# 2. Joshimath
# =========================================================

joshimath = ee.Geometry.Point([
    79.56436,   # longitude
    30.55543    # latitude
])

# 10 km study area
roi = joshimath.buffer(10000)

print("✅ Joshimath 10 km study area created")


# =========================================================
# 3. Global Flood Database
# =========================================================

gfd = ee.ImageCollection(
    "GLOBAL_FLOOD_DB/MODIS_EVENTS/V1"
)

print("✅ Global Flood Database loaded")


# =========================================================
# 4. Output folder
# =========================================================

output_folder = "data/historical_flood_monthly"

os.makedirs(
    output_folder,
    exist_ok=True
)


# =========================================================
# 5. Process every month
# =========================================================

for year in range(2000, 2019):

    for month in range(1, 13):

        # GFD starts on 17 February 2000
        if year == 2000 and month == 1:
            continue

        print(f"\n📥 Processing {year}-{month:02d}...")


        # -------------------------------------------------
        # Month start/end
        # -------------------------------------------------

        start = ee.Date.fromYMD(
            year,
            month,
            1
        )

        end = start.advance(
            1,
            "month"
        )


        # -------------------------------------------------
        # Filter GFD by month
        # -------------------------------------------------

        monthly = (
            gfd
            .filterDate(start, end)
        )


        # -------------------------------------------------
        # Number of flood events
        # -------------------------------------------------

        event_count = monthly.size().getInfo()

        print(
            f"   GFD events in month: {event_count}"
        )


        # -------------------------------------------------
        # Find events that actually overlap Joshimath
        # -------------------------------------------------

        def process_event(image):

            # Flooded pixels inside Joshimath area
            flooded = (
                image
                .select("flooded")
                .reduceRegion(
                    reducer=ee.Reducer.sum(),
                    geometry=roi,
                    scale=30,
                    maxPixels=1e9
                )
                .get("flooded")
            )


            # Duration
            duration = (
                image
                .select("duration")
                .reduceRegion(
                    reducer=ee.Reducer.max(),
                    geometry=roi,
                    scale=30,
                    maxPixels=1e9
                )
                .get("duration")
            )


            return ee.Feature(
                None,
                {
                    "event_id": image.get("id"),

                    "flooded_pixels": flooded,

                    "max_flood_duration_days": duration,

                    "dfo_country": image.get(
                        "dfo_country"
                    ),

                    "dfo_main_cause": image.get(
                        "dfo_main_cause"
                    ),

                    "dfo_severity": image.get(
                        "dfo_severity"
                    ),

                    "dfo_dead": image.get(
                        "dfo_dead"
                    ),

                    "dfo_displaced": image.get(
                        "dfo_displaced"
                    ),

                    "dfo_centroid_x": image.get(
                        "dfo_centroid_x"
                    ),

                    "dfo_centroid_y": image.get(
                        "dfo_centroid_y"
                    ),

                    "clear_views": image.get(
                        "clear_views"
                    ),

                    "clear_perc": image.get(
                        "clear_perc"
                    ),

                    "year": year,

                    "month": month
                }
            )


        # -------------------------------------------------
        # Process events
        # -------------------------------------------------

        features = monthly.map(
            process_event
        )


        # -------------------------------------------------
        # Keep only events with flood pixels
        # -------------------------------------------------

        features = features.filter(
            ee.Filter.gt(
                "flooded_pixels",
                0
            )
        )


        # -------------------------------------------------
        # Get data
        # -------------------------------------------------

        data = features.getInfo()


        # -------------------------------------------------
        # Convert to Python
        # -------------------------------------------------

        rows = []

        for feature in data["features"]:

            props = feature["properties"]

            rows.append({

                "event_id":
                    props.get("event_id"),

                "flooded_pixels":
                    props.get("flooded_pixels"),

                "max_flood_duration_days":
                    props.get(
                        "max_flood_duration_days"
                    ),

                "dfo_country":
                    props.get("dfo_country"),

                "dfo_main_cause":
                    props.get(
                        "dfo_main_cause"
                    ),

                "dfo_severity":
                    props.get(
                        "dfo_severity"
                    ),

                "dfo_dead":
                    props.get("dfo_dead"),

                "dfo_displaced":
                    props.get(
                        "dfo_displaced"
                    ),

                "dfo_centroid_x":
                    props.get(
                        "dfo_centroid_x"
                    ),

                "dfo_centroid_y":
                    props.get(
                        "dfo_centroid_y"
                    ),

                "clear_views":
                    props.get("clear_views"),

                "clear_perc":
                    props.get("clear_perc"),

                "year":
                    year,

                "month":
                    month,

                "flood_occurrence":
                    1

            })


        # -------------------------------------------------
        # Create DataFrame
        # -------------------------------------------------

        df = pd.DataFrame(rows)


        # -------------------------------------------------
        # If no flood event exists
        # -------------------------------------------------

        if df.empty:

            df = pd.DataFrame([{

                "event_id": None,

                "flooded_pixels": 0,

                "max_flood_duration_days": 0,

                "dfo_country": None,

                "dfo_main_cause": None,

                "dfo_severity": None,

                "dfo_dead": None,

                "dfo_displaced": None,

                "dfo_centroid_x": None,

                "dfo_centroid_y": None,

                "clear_views": None,

                "clear_perc": None,

                "year": year,

                "month": month,

                "flood_occurrence": 0

            }])


        # -------------------------------------------------
        # Save monthly CSV
        # -------------------------------------------------

        filename = (
            f"{output_folder}/"
            f"flood_{year}_{month:02d}.csv"
        )


        df.to_csv(
            filename,
            index=False
        )


        print(
            f"   ✅ Saved: {filename}"
        )

        print(
            f"   Flood events in Joshimath: "
            f"{len(df[df['flood_occurrence'] == 1])}"
        )


print(
    "\n🎉 ALL HISTORICAL FLOOD MONTHLY FILES COMPLETED!"
)

In [ ]:
# ============================================================
# GENERATE MONTH-BY-MONTH LANDSLIDE CSV FILES
# ============================================================

output_folder = os.path.join(
    PROJECT_FOLDER,
    "data",
    "historical_landslide_monthly"
)

os.makedirs(
    output_folder,
    exist_ok=True
)


print("\n============================================================")
print(" MONTH-BY-MONTH LANDSLIDE EXTRACTION")
print("============================================================")


# ------------------------------------------------------------
# Historical period
# ------------------------------------------------------------

START_YEAR = 2007
END_YEAR = 2018


# ------------------------------------------------------------
# Process every year
# ------------------------------------------------------------

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    # --------------------------------------------------------
    # Process every month
    # --------------------------------------------------------

    for month in range(
        1,
        13
    ):

        print(
            f"\n📥 Processing: {year}-{month:02d}"
        )


        # ====================================================
        # SELECT ONLY THIS MONTH
        # ====================================================

        monthly = landslide_joshimath[
            (
                landslide_joshimath["year"]
                == year
            )
            &
            (
                landslide_joshimath["month"]
                == month
            )
        ].copy()


        # ====================================================
        # COUNT LANDSLIDES
        # ====================================================

        landslide_count = len(
            monthly
        )


        print(
            "   Landslides found:",
            landslide_count
        )


        # ====================================================
        # CREATE MONTHLY DATA
        # ====================================================

        rows = []


        # ====================================================
        # IF LANDSLIDE EXISTS
        # ====================================================

        if landslide_count > 0:

            for index, event in monthly.iterrows():

                rows.append({

                    "event_id":
                        event.get(
                            "event_id",
                            index
                        ),

                    "event_date":
                        event.get(
                            "event_date"
                        ),

                    "latitude":
                        event.get(
                            LAT_COLUMN
                        ),

                    "longitude":
                        event.get(
                            LON_COLUMN
                        ),

                    "year":
                        year,

                    "month":
                        month,

                    "landslide_count":
                        landslide_count,

                    "landslide_occurrence":
                        1

                })


        # ====================================================
        # IF NO LANDSLIDE EXISTS
        # ====================================================

        else:

            rows.append({

                "event_id":
                    None,

                "event_date":
                    None,

                "latitude":
                    None,

                "longitude":
                    None,

                "year":
                    year,

                "month":
                    month,

                "landslide_count":
                    0,

                "landslide_occurrence":
                    0

            })


        # ====================================================
        # CREATE DATAFRAME
        # ====================================================

        df = pd.DataFrame(
            rows
        )


        # ====================================================
        # SAVE MONTHLY CSV
        # ====================================================

        filename = os.path.join(

            output_folder,

            f"landslide_{year}_{month:02d}.csv"

        )


        df.to_csv(

            filename,

            index=False

        )


        # ====================================================
        # PRINT RESULT
        # ====================================================

        print(
            "   ✅ Saved:",
            filename
        )

        print(
            "   📊 Occurrence:",
            1 if landslide_count > 0 else 0
        )


# ============================================================
# FINAL CHECK
# ============================================================

monthly_files = sorted(

    glob.glob(

        os.path.join(

            output_folder,

            "landslide_*.csv"

        )

    )

)


print("\n============================================================")
print(" 🎉 MONTHLY LANDSLIDE DATA COMPLETED")
print("============================================================")

print(
    "Total monthly files:",
    len(monthly_files)
)

print(
    "Expected:",
    (END_YEAR - START_YEAR + 1) * 12
)

print(
    "\nOutput folder:"
)

print(
    output_folder
)

In [ ]:
!pip install pandas numpy matplotlib scikit-learn xgboost torch statsmodels

In [ ]:
import zipfile
import os

zip_path = r"E:\MY projects\Flash Flood Prediction Project\data\rainfall_monthly.zip"
extract_path = r"E:\MY projects\Flash Flood Prediction Project\data\rainfall_monthly"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully!")

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    print("GPU not available - using CPU")
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
import glob

csv_files = glob.glob(
    os.path.join(extract_path, "**", "*.csv"),
    recursive=True
)

print("Number of CSV files:", len(csv_files))
print(csv_files[:5])

In [ ]:
data_list = []

for file in csv_files:
    temp = pd.read_csv(file)
    data_list.append(temp)

rainfall_df = pd.concat(data_list, ignore_index=True)

print("Shape:", rainfall_df.shape)
print(rainfall_df.head())
print(rainfall_df.tail())

In [ ]:
rainfall_df["datetime"] = pd.to_datetime(
    rainfall_df["datetime"],
    errors="coerce"
)

rainfall_df["rainfall_mm_per_hour"] = pd.to_numeric(
    rainfall_df["rainfall_mm_per_hour"],
    errors="coerce"
)

# Remove invalid rows
rainfall_df = rainfall_df.dropna(
    subset=["datetime", "rainfall_mm_per_hour"]
)

# Remove infinite values
rainfall_df = rainfall_df[
    np.isfinite(rainfall_df["rainfall_mm_per_hour"])
]

# Rainfall cannot be negative
rainfall_df["rainfall_mm_per_hour"] = rainfall_df[
    "rainfall_mm_per_hour"
].clip(lower=0)

# Remove duplicate timestamps
rainfall_df = rainfall_df.drop_duplicates(
    subset=["datetime"]
)

# Sort chronologically
rainfall_df = rainfall_df.sort_values("datetime")

rainfall_df = rainfall_df.reset_index(drop=True)

print(rainfall_df.info())
print(rainfall_df.describe())

In [ ]:
print("NaN values:")
print(rainfall_df.isna().sum())

print("\nInfinite values:")
print(np.isinf(
    rainfall_df["rainfall_mm_per_hour"]
).sum())

In [ ]:
rainfall_df = rainfall_df.set_index("datetime")

hourly_rainfall = rainfall_df[
    "rainfall_mm_per_hour"
].resample("1h").sum()

hourly_rainfall = hourly_rainfall.to_frame(
    name="rainfall"
)

hourly_rainfall = hourly_rainfall.dropna()

print(hourly_rainfall.head())
print(hourly_rainfall.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

plt.plot(
    hourly_rainfall.index,
    hourly_rainfall["rainfall"]
)

plt.title("Hourly Rainfall")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

hourly_rainfall["rainfall"].tail(24 * 90).plot()

plt.title("Rainfall - Recent 90 Days")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")

plt.show()

In [ ]:
df = hourly_rainfall.copy()

# Lag features
for lag in [1, 2, 3, 6, 12, 24, 48]:
    df[f"lag_{lag}"] = df["rainfall"].shift(lag)

# Rolling rainfall
df["rolling_3h"] = df["rainfall"].rolling(3).sum()
df["rolling_6h"] = df["rainfall"].rolling(6).sum()
df["rolling_12h"] = df["rainfall"].rolling(12).sum()
df["rolling_24h"] = df["rainfall"].rolling(24).sum()

# Time features
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

# Target = next hour rainfall
df["target"] = df["rainfall"].shift(-1)

df = df.dropna()

print(df.head())
print("Final shape:", df.shape)

In [ ]:
feature_columns = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",
    "lag_24",a
    "lag_48",
    "rolling_3h",
    "rolling_6h",
    "rolling_12h",
    "rolling_24h",
    "hour",
    "dayofweek",
    "month"
]

X = df[feature_columns].values
y = df["target"].values

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
split_index = int(len(X) * 0.80)

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
).ravel()

y_test_scaled = y_scaler.transform(
    y_test.reshape(-1, 1)
).ravel()

print(
    np.isfinite(X_train_scaled).all(),
    np.isfinite(y_train_scaled).all()
)

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    device="cuda",
    random_state=42
)

xgb_model.fit(
    X_train_scaled,
    y_train_scaled,
    eval_set=[
        (X_test_scaled, y_test_scaled)
    ],
    verbose=False
)

print("XGBoost GPU training completed.")

In [ ]:
xgb_pred_scaled = xgb_model.predict(X_test_scaled)

xgb_pred = y_scaler.inverse_transform(
    xgb_pred_scaled.reshape(-1, 1)
).ravel()

# Rainfall cannot be negative
xgb_pred = np.clip(xgb_pred, 0, None)

print(xgb_pred[:10])

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def calculate_metrics(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    # Avoid division by zero in rainfall data
    mask = y_true != 0

    if mask.sum() > 0:
        mape = np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        ) * 100
    else:
        mape = np.nan

    r2 = r2_score(y_true, y_pred)

    return mae, rmse, mape, r2


xgb_metrics = calculate_metrics(
    y_test,
    xgb_pred
)

print("XGBoost")
print("MAE  :", xgb_metrics[0])
print("RMSE :", xgb_metrics[1])
print("MAPE :", xgb_metrics[2])
print("R2   :", xgb_metrics[3])

In [ ]:
sequence_length = 24

rainfall_values = hourly_rainfall["rainfall"].values

# Scale rainfall
rainfall_scaler = StandardScaler()

rainfall_scaled = rainfall_scaler.fit_transform(
    rainfall_values.reshape(-1, 1)
).flatten()

X_seq = []
y_seq = []

for i in range(
    sequence_length,
    len(rainfall_scaled)
):

    X_seq.append(
        rainfall_scaled[
            i-sequence_length:i
        ]
    )

    y_seq.append(
        rainfall_scaled[i]
    )

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print("Sequence shape:", X_seq.shape)
print("Target shape:", y_seq.shape)

In [ ]:
X_seq = X_seq.reshape(
    X_seq.shape[0],
    X_seq.shape[1],
    1
)

print(X_seq.shape)

In [ ]:
split = int(len(X_seq) * 0.80)

X_train_seq = X_seq[:split]
X_test_seq = X_seq[split:]

y_train_seq = y_seq[:split]
y_test_seq = y_seq[split:]

print(X_train_seq.shape)
print(X_test_seq.shape)

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(
    X_train_seq,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_seq,
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test_seq,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test_seq,
    dtype=torch.float32
).reshape(-1, 1)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=False
)

print("Training samples:", len(train_dataset))

In [ ]:
import torch.nn as nn

class RainfallLSTM(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=64,
        num_layers=2
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        output, _ = self.lstm(x)

        last_output = output[:, -1, :]

        return self.fc(last_output)

In [ ]:
lstm_model = RainfallLSTM().to(device)

print(lstm_model)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

epochs = 50

for epoch in range(epochs):

    lstm_model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = lstm_model(X_batch)

        loss = criterion(
            predictions,
            y_batch
        )

        # Safety check against NaN
        if not torch.isfinite(loss):
            print("NaN/Inf loss detected!")
            break

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            lstm_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {avg_loss:.6f}"
        )

In [ ]:
lstm_model.eval()

with torch.no_grad():

    X_test_gpu = X_test_tensor.to(device)

    lstm_pred_scaled = lstm_model(
        X_test_gpu
    )

    lstm_pred_scaled = (
        lstm_pred_scaled
        .cpu()
        .numpy()
        .flatten()
    )

lstm_pred = rainfall_scaler.inverse_transform(
    lstm_pred_scaled.reshape(-1, 1)
).flatten()

lstm_actual = rainfall_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).flatten()

lstm_pred = np.clip(
    lstm_pred,
    0,
    None
)

print(lstm_pred[:10])

In [ ]:
lstm_metrics = calculate_metrics(
    lstm_actual,
    lstm_pred
)

print("LSTM")
print("MAE  :", lstm_metrics[0])
print("RMSE :", lstm_metrics[1])
print("MAPE :", lstm_metrics[2])
print("R2   :", lstm_metrics[3])

In [ ]:
class RainfallGRU(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=64,
        num_layers=2
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        output, _ = self.gru(x)

        last_output = output[:, -1, :]

        return self.fc(last_output)

In [ ]:
gru_model = RainfallGRU().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    gru_model.parameters(),
    lr=0.001
)

epochs = 50

for epoch in range(epochs):

    gru_model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = gru_model(X_batch)

        loss = criterion(
            predictions,
            y_batch
        )

        if not torch.isfinite(loss):
            print("NaN/Inf loss detected!")
            break

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            gru_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {avg_loss:.6f}"
        )

In [ ]:
gru_model.eval()

with torch.no_grad():

    X_test_gpu = X_test_tensor.to(device)

    gru_pred_scaled = gru_model(
        X_test_gpu
    )

    gru_pred_scaled = (
        gru_pred_scaled
        .cpu()
        .numpy()
        .flatten()
    )

gru_pred = rainfall_scaler.inverse_transform(
    gru_pred_scaled.reshape(-1, 1)
).flatten()

gru_actual = rainfall_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).flatten()

gru_pred = np.clip(
    gru_pred,
    0,
    None
)

In [ ]:
gru_metrics = calculate_metrics(
    gru_actual,
    gru_pred
)

print("GRU")
print("MAE  :", gru_metrics[0])
print("RMSE :", gru_metrics[1])
print("MAPE :", gru_metrics[2])
print("R2   :", gru_metrics[3])

In [ ]:
results = pd.DataFrame({

    "Model": [
        "XGBoost",
        "LSTM",
        "GRU"
    ],

    "MAE": [
        xgb_metrics[0],
        lstm_metrics[0],
        gru_metrics[0]
    ],

    "RMSE": [
        xgb_metrics[1],
        lstm_metrics[1],
        gru_metrics[1]
    ],

    "MAPE": [
        xgb_metrics[2],
        lstm_metrics[2],
        gru_metrics[2]
    ],

    "R2": [
        xgb_metrics[3],
        lstm_metrics[3],
        gru_metrics[3]
    ]
})

results

In [ ]:
results["MAE_rank"] = results["MAE"].rank()
results["RMSE_rank"] = results["RMSE"].rank()
results["MAPE_rank"] = results["MAPE"].rank()
results["R2_rank"] = results["R2"].rank(
    ascending=False
)

results["Total_rank"] = (
    results["MAE_rank"]
    + results["RMSE_rank"]
    + results["MAPE_rank"]
    + results["R2_rank"]
)

results = results.sort_values(
    "Total_rank"
)

print(results)

best_model = results.iloc[0]["Model"]

print("\nBEST RAINFALL FORECASTING MODEL:")
print(best_model)

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    y_test,
    label="Actual"
)

plt.plot(
    xgb_pred,
    label="XGBoost"
)

plt.title("Actual vs XGBoost Rainfall Forecast")
plt.xlabel("Time")
plt.ylabel("Rainfall")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    lstm_actual,
    label="Actual"
)

plt.plot(
    lstm_pred,
    label="LSTM"
)

plt.title("Actual vs LSTM Rainfall Forecast")
plt.xlabel("Time")
plt.ylabel("Rainfall")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    gru_actual,
    label="Actual"
)

plt.plot(
    gru_pred,
    label="GRU"
)

plt.title("Actual vs GRU Rainfall Forecast")
plt.xlabel("Time")
plt.ylabel("Rainfall")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    results["Model"],
    results["RMSE"]
)

plt.title("Rainfall Forecasting RMSE Comparison")
plt.xlabel("Model")
plt.ylabel("RMSE")

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    results["Model"],
    results["R2"]
)

plt.title("Rainfall Forecasting R² Comparison")
plt.xlabel("Model")
plt.ylabel("R²")

plt.show()

In [ ]:
def forecast_next_hours(
    model,
    recent_values,
    hours=24
):

    model.eval()

    sequence = recent_values.copy()

    forecasts = []

    for _ in range(hours):

        input_data = np.array(
            sequence[-24:]
        )

        input_scaled = rainfall_scaler.transform(
            input_data.reshape(-1, 1)
        )

        tensor = torch.tensor(
            input_scaled,
            dtype=torch.float32
        ).reshape(1, 24, 1)

        tensor = tensor.to(device)

        with torch.no_grad():

            prediction = model(
                tensor
            )

        prediction = prediction.cpu().numpy()[0][0]

        prediction_original = (
            rainfall_scaler.inverse_transform(
                [[prediction]]
            )[0][0]
        )

        prediction_original = max(
            0,
            prediction_original
        )

        forecasts.append(
            prediction_original
        )

        sequence = np.append(
            sequence,
            prediction_original
        )

    return np.array(forecasts)

In [ ]:
recent_values = hourly_rainfall[
    "rainfall"
].values[-24:]

future_forecast = forecast_next_hours(
    lstm_model,
    recent_values,
    hours=24
)

print("Next 24-hour rainfall forecast:")
print(future_forecast)

In [ ]:
last_time = hourly_rainfall.index[-1]

future_times = pd.date_range(
    start=last_time + pd.Timedelta(hours=1),
    periods=24,
    freq="1h"
)

plt.figure(figsize=(15, 5))

plt.plot(
    future_times,
    future_forecast,
    marker="o"
)

plt.title(
    "Next 24 Hours Rainfall Forecast"
)

plt.xlabel("Time")
plt.ylabel("Forecast Rainfall (mm)")

plt.xticks(rotation=45)
plt.grid(True)

plt.show()

In [ ]:
max_rainfall = future_forecast.max()

total_24h_rainfall = future_forecast.sum()

print("Maximum hourly rainfall:",
      max_rainfall)

print("Forecasted 24-hour rainfall:",
      total_24h_rainfall)

In [ ]:
rainfall_risk_score = min(
    total_24h_rainfall / 200,
    1.0
)

print(
    "Rainfall Risk Score:",
    rainfall_risk_score
)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

In [ ]:
series = hourly_rainfall["rainfall"].copy()

series = series.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

print(series.head())
print("Number of observations:", len(series))

In [ ]:
split_index = int(len(series) * 0.80)

train = series.iloc[:split_index]
test = series.iloc[split_index:]

print("Training samples:", len(train))
print("Testing samples :", len(test))

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(
    train,
    order=(1, 1, 1)
)

arima_result = arima_model.fit()

print(arima_result.summary())

In [ ]:
arima_forecast = arima_result.forecast(
    steps=len(test)
)

arima_forecast = arima_forecast.clip(
    lower=0
)

print(arima_forecast.head())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# --------------------------------
# 1. Prepare rainfall series
# --------------------------------

series = hourly_rainfall["rainfall"].copy()

series = series.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

# Make sure index is sorted
series = series.sort_index()

print("Total observations:", len(series))

# --------------------------------
# 2. Train/Test split
# --------------------------------

split_index = int(len(series) * 0.80)

train = series.iloc[:split_index]
test = series.iloc[split_index:]

print("Train:", len(train))
print("Test :", len(test))

# --------------------------------
# 3. ARIMA
# --------------------------------

print("\nTraining ARIMA...")

arima_model = ARIMA(
    train,
    order=(1, 1, 1)
)

arima_result = arima_model.fit()

print("ARIMA training completed!")

# --------------------------------
# 4. Forecast
# --------------------------------

arima_forecast = arima_result.forecast(
    steps=len(test)
)

# Rainfall cannot be negative
arima_forecast = np.maximum(
    np.asarray(arima_forecast),
    0
)

# --------------------------------
# 5. Display forecast
# --------------------------------

print("\nFirst 10 predictions:")

for i in range(min(10, len(arima_forecast))):
    print(
        f"{test.index[i]} -> "
        f"{arima_forecast[i]:.4f} mm"
    )

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

actual = test.values
predicted = arima_forecast

mae = mean_absolute_error(
    actual,
    predicted
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        predicted
    )
)

# MAPE excluding zero rainfall
mask = actual != 0

if mask.sum() > 0:
    mape = np.mean(
        np.abs(
            (actual[mask] - predicted[mask])
            / actual[mask]
        )
    ) * 100
else:
    mape = np.nan

r2 = r2_score(
    actual,
    predicted
)

print("\n==============================")
print("       ARIMA RESULTS")
print("==============================")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.4f}%")
print(f"R²   : {r2:.4f}")

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    test.index,
    test.values,
    label="Actual"
)

plt.plot(
    test.index,
    arima_forecast,
    label="ARIMA Forecast"
)

plt.title(
    "ARIMA Rainfall Forecast"
)

plt.xlabel("Time")
plt.ylabel("Rainfall (mm)")

plt.legend()
plt.grid(True)

plt.show()